# Kv2.1 WT: maskV1 versus maskV2

Comparison of the two supplied maskV2 distance exports with the original maskV1 ensemble. The notebook combines the exports without double counting, reproduces the regional distance-distribution approach used for Figure 3, and quantifies structural sampling breadth globally and separately in the voltage-sensor domain (VSD), selectivity filter, and S6/intracellular gate.

**Primary comparison:** filtered maskV1 versus filtered maskV2, using recycle 10 and the same `(seed, AF2 model)` trajectories. Both masks use their final all_ok3, structural, interface, and alignment QC selections. Differences describe these prediction ensembles; they do not establish conformational dynamics or a causal effect of mask placement alone.

Run all cells from the repository or this notebook's directory. Dependencies: Python, NumPy, pandas, SciPy, Matplotlib, seaborn, Biopython, nbformat, and the repository's data-access module. Large baseline inputs are resolved locally or from the companion dataset. All output paths are relative to the repository.

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib, json, os, re, sys
import numpy as np
import pandas as pd
import scipy
from scipy.stats import wasserstein_distance
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'shared/data_access.py').is_file())
sys.path.insert(0, str(ROOT))
from shared import data_access
OUT = ROOT / 'kv21/dataDistances/analysis/mask_v1_vs_v2_filtered'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
for directory in [OUT, FIG, TAB]:
    directory.mkdir(parents=True, exist_ok=True)
RNG_SEED = 20261011
N_BOOT = 5000
COLORS = {'maskV1': '#3B6F8F', 'maskV2': '#D77B32'}
REGIONS = ['Voltage sensors', 'Selectivity filter', 'S6 / intracellular gate']
sns.set_theme(style='ticks', context='notebook')
matplotlib.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 220, 'pdf.fonttype': 42,
                           'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                           'axes.spines.right': False})
pd.set_option('display.max_columns', 16)
def savefig(fig, name):
    for suffix in ['png', 'pdf']:
        fig.savefig(FIG / f'{name}.{suffix}', bbox_inches='tight')
    plt.show()
    plt.close(fig)
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

## Input reconciliation and provenance

Files with the same structure filename must agree in every distance value; conflicting records stop execution. Exact duplicate structures are retained once, with both source filenames recorded. Unsuffixed final PDB rows are checked against recycle 10 and excluded from recycle-based analyses to avoid counting the same prediction twice. No arbitrary seed offset or new batch identity is assigned to duplicate exports.

In [ ]:
v1_rel = 'kv21/dataDistances/26-02-11_Kv2.1_wt_maskedAF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'
v2_rel = 'kv21/dataDistances/26-10-09_Kv2.1_wt_masked_v2_AF2_distances_all_ok_rmsd_3A_structural_interface_alignment_qc.csv'
v1_path = data_access.resolve(v1_rel)
v2_path = ROOT / v2_rel
assert v2_path.is_file(), v2_path
raw = {'maskV1': pd.read_csv(v1_path), 'maskV2': pd.read_csv(v2_path)}
qc_raw = raw['maskV1'].copy()
manifest = pd.DataFrame([
    {'role': 'maskV1 filtered', 'repository_path': v1_rel, 'sha256': sha256(v1_path), 'rows': len(raw['maskV1']), 'columns': len(raw['maskV1'].columns), 'bytes': v1_path.stat().st_size},
    {'role': 'maskV2 filtered', 'repository_path': v2_rel, 'sha256': sha256(v2_path), 'rows': len(raw['maskV2']), 'columns': len(raw['maskV2'].columns), 'bytes': v2_path.stat().st_size},
])
manifest.to_csv(TAB / 'input_manifest.csv', index=False)
print(manifest.to_string(index=False))
print(f"Filtered inputs: maskV1 {len(raw['maskV1']):,} rows; maskV2 {len(raw['maskV2']):,} rows.")


In [ ]:
def metadata(frame):
    parsed = frame.pdb_file.str.extract(r'_model_(?P<model>\d+)_seed_(?P<seed>\d+)(?:\.r(?P<recycle>\d+))?\.pdb$')
    assert parsed[['model', 'seed']].notna().all().all(), 'Unparsed structure identifiers.'
    parsed[['model', 'seed']] = parsed[['model', 'seed']].astype(int)
    parsed['recycle'] = parsed.recycle.fillna(-1).astype(int)
    frame = frame.drop(columns=[c for c in ['model', 'seed', 'recycle'] if c in frame], errors='ignore')
    return pd.concat([frame.reset_index(drop=True), parsed.reset_index(drop=True)], axis=1)

cohort_rows = []
for label, frame in list(raw.items()) + [('maskV1 published QC', qc_raw)]:
    frame = metadata(frame)
    assert not frame.duplicated(['seed', 'model', 'recycle']).any()
    distance_cols = [c for c in frame if c.startswith(('CA_', 'shortest_'))]
    values = frame[distance_cols].to_numpy(float)
    assert np.isfinite(values).all() and (values >= 0).all()
    final = frame[frame.recycle.eq(-1)].set_index(['seed', 'model'])
    r10 = frame[frame.recycle.eq(10)].set_index(['seed', 'model'])
    if len(final):
        assert final.index.equals(r10.index) or set(final.index) == set(r10.index)
        np.testing.assert_array_equal(final[distance_cols].sort_index(), r10[distance_cols].sort_index())
    frame = frame[frame.recycle.ge(0)].copy()
    if label in raw:
        raw[label] = frame
    else:
        qc_raw = frame
    cohort_rows.append({'cohort': label, 'explicit_recycle_rows': len(frame),
                       'final_alias_rows_excluded': len(final), 'seeds': frame.seed.nunique(),
                       'trajectories': frame[['seed', 'model']].drop_duplicates().shape[0],
                       'recycles': ', '.join(map(str, sorted(frame.recycle.unique())))})
cohorts = pd.DataFrame(cohort_rows)
cohorts.to_csv(TAB / 'cohort_counts.csv', index=False)
print(cohorts.to_string(index=False))
common = sorted(set(raw['maskV1']) & set(raw['maskV2']))
shared_distances = [c for c in common if c.startswith(('CA_', 'shortest_'))]
coverage = pd.DataFrame({'column': sorted(set(raw['maskV1']) | set(raw['maskV2']))})
coverage = coverage[coverage.column.str.startswith(('CA_', 'shortest_'))].copy()
for mask in raw:
    coverage[mask] = coverage.column.isin(raw[mask].columns)
coverage.to_csv(TAB / 'distance_column_coverage.csv', index=False)
print(f'{len(shared_distances):,} shared raw distance columns; '
      f'{sum(coverage.maskV1 & ~coverage.maskV2)} maskV1-only columns.')

## Coordinate definitions and anatomical coverage

The primary breadth analysis uses **Cα distances**. Minimum interatomic distances are summarized separately as a sensitivity analysis. Within each structure, the four homologous intrachain VSD measurements are sorted, and each complete same-residue six-distance pore ring is sorted. These order statistics retain subunit asymmetry while remaining invariant to chain relabeling. Each rank is a separate coordinate. The maximum ring span is also used for the S6 profile, as in the existing Kv2.1 notebook; it is a Cα diameter proxy, not a solvent-accessible pore radius.

The complete common panel contains 56 VSD residue pairs, four selectivity-filter rings, and four S6 rings, for each measurement type. Other interchain columns are excluded from breadth because the supplied tables do not contain all ordered chain assignments needed for an invariant cross-residue comparison. Exact source-column membership is exported.

Plot labels use the existing rat experimental/paper numbering (**CSV residue − 2**). Model/CSV S6 positions 407 and 411 (paper I405 and V409) lack complete rings in maskV2 and cannot be assessed. Region labels describe measured anatomy; exact maskV2 residue placement cannot be reconstructed from distance CSVs alone.

In [ ]:
ca_pattern = re.compile(r'^CA_CA_([ABCD])_([A-Z]{3})(\d+)_CA-([ABCD])_([A-Z]{3})(\d+)_CA$')
short_pattern = re.compile(r'^shortest_([A-Z]{3})(\d+)([ABCD])-([A-Z]{3})(\d+)([ABCD])$')
aa = {'ARG': 'R', 'LYS': 'K', 'GLU': 'E', 'ASP': 'D', 'CYS': 'C', 'PHE': 'F',
      'VAL': 'V', 'ILE': 'I', 'ALA': 'A', 'LEU': 'L', 'THR': 'T', 'GLY': 'G', 'TYR': 'Y'}
groups = {}
for column in shared_distances:
    m = ca_pattern.fullmatch(column)
    if m:
        ch1, res1, n1, ch2, res2, n2 = m.groups(); kind = 'CA'
    else:
        m = short_pattern.fullmatch(column)
        assert m, column
        res1, n1, ch1, res2, n2, ch2 = m.groups(); kind = 'shortest'
    n1, n2 = int(n1), int(n2)
    if ch1 == ch2 and 170 <= n1 <= 320 and 170 <= n2 <= 320:
        region, expected = REGIONS[0], 4
    elif ch1 != ch2 and (res1, n1) == (res2, n2) and 375 <= n1 <= 379:
        region, expected = REGIONS[1], 6
    elif ch1 != ch2 and (res1, n1) == (res2, n2) and 400 <= n1 <= 411:
        region, expected = REGIONS[2], 6
    else:
        continue
    key = (kind, region, res1, n1, res2, n2, expected)
    groups.setdefault(key, []).append(column)

coordinate_rows = []
for (kind, region, r1, n1, r2, n2, expected), cols in sorted(groups.items()):
    assert len(cols) == expected, (kind, region, cols)
    pair = f'{aa[r1]}{n1 - 2}–{aa[r2]}{n2 - 2}' if n1 != n2 else f'{aa[r1]}{n1 - 2}'
    for rank in range(1, expected + 1):
        coordinate_rows.append({'coordinate': f'{kind}_{r1}{n1}_{r2}{n2}_rank{rank}',
            'measurement': kind, 'region': region, 'pair': pair, 'rank': rank,
            'rank_count': expected, 'csv_residue_1': n1, 'csv_residue_2': n2,
            'paper_residue_1': aa[r1] + str(n1 - 2), 'paper_residue_2': aa[r2] + str(n2 - 2),
            'source_columns': ';'.join(sorted(cols))})
coordinates = pd.DataFrame(coordinate_rows)
coordinates.to_csv(TAB / 'coordinate_definitions.csv', index=False)
primary_cols = coordinates.loc[coordinates.measurement.eq('CA'), 'coordinate'].tolist()
region_cols = {r: coordinates.loc[coordinates.measurement.eq('CA') & coordinates.region.eq(r), 'coordinate'].tolist()
               for r in REGIONS}

def transform(frame):
    arrays = {}
    for _, rows in coordinates.groupby('source_columns', sort=False):
        values = np.sort(frame[rows.iloc[0].source_columns.split(';')].to_numpy(float), axis=1)
        for row in rows.itertuples():
            arrays[row.coordinate] = values[:, row.rank - 1]
    return pd.concat([frame[['pdb_file', 'seed', 'model', 'recycle']].reset_index(drop=True),
                      pd.DataFrame(arrays)], axis=1)
transformed = {k: transform(v) for k, v in raw.items()}
qc_transformed = transform(qc_raw)
print(coordinates.groupby(['measurement', 'region']).agg(coordinates=('coordinate', 'size'),
          residue_pairs=('pair', 'nunique')).reset_index().to_string(index=False))
print('Raw columns used:', len(set(';'.join(coordinates.source_columns).split(';'))),
      '/', len(shared_distances))

## Equal-depth primary cohort

Each mask contributes one recycle-10 structure per shared `(seed, AF2 model)` trajectory. Nominal seed labels are used as computational blocks, not biological replicates. Pairing controls sampling depth and AF2 model composition; matching numeric seeds does not establish identical stochastic sampling paths across masking runs. No recycle snapshots or subunits are treated as independent replicates.

In [ ]:
def matched_pair(frames, recycle=10):
    selected = {k: f[f.recycle.eq(recycle)].set_index(['seed', 'model']).sort_index()
                for k, f in frames.items()}
    common_keys = selected['maskV1'].index.intersection(selected['maskV2'].index)
    assert len(common_keys), 'No matched trajectories.'
    return {k: f.loc[common_keys].reset_index() for k, f in selected.items()}
primary = matched_pair(transformed)
assert primary['maskV1'][['seed', 'model']].equals(primary['maskV2'][['seed', 'model']])
for frame in primary.values():
    assert frame.groupby('seed').model.nunique().ge(1).all()
primary_counts = pd.DataFrame([{'mask': k, 'structures': len(f), 'seeds': f.seed.nunique(),
    'AF2_models_per_seed': f.groupby('seed').model.nunique().min(), 'recycle': 10}
    for k, f in primary.items()])
print(primary_counts.to_string(index=False))
primary_counts.to_csv(TAB / 'primary_cohort_counts.csv', index=False)
for mask, frame in primary.items():
    frame.to_csv(OUT / f'{mask}_matched_r10_coordinates.csv.gz', index=False,
                 compression={'method': 'gzip', 'mtime': 0})

## Sampling breadth: global and by channel region

For coordinate \(j\), the scale is its IQR pooled over the two equally sized recycle-10 ensembles. Within seed \(s\), breadth is the IQR across its the available retained AF2 models divided by this common scale. The median across eligible coordinates gives the seed's global or regional breadth. Zero pooled-IQR coordinates are excluded explicitly. Global breadth refers to the **available measured panel**, not the entire protein; VSD coordinates are more numerous. A region-balanced summary gives the three regions equal weight.

The reported ratio is median seed breadth(maskV2) / median seed breadth(maskV1). Paired whole-seed bootstrap intervals resample shared seed labels 5,000 times, holding the pooled coordinate scales fixed. An independent-seed bootstrap is included to test sensitivity to nominal-seed pairing. These intervals describe variation over seeds conditional on the five model parameterizations and the supplied ensembles; they do not address the missing structural-QC measurements.

In [ ]:
all_cols = coordinates.coordinate.tolist()
pooled = pd.concat([primary[k][all_cols] for k in COLORS], ignore_index=True)
scale = pooled.quantile(.75) - pooled.quantile(.25)
valid_cols = scale[scale.gt(0)].index.tolist()
scale.rename('pooled_IQR_A').rename_axis('coordinate').reset_index().to_csv(TAB / 'normalization_scales.csv', index=False)
print('Zero-IQR coordinates excluded:', int(scale.eq(0).sum()))

breadth_rows = []
for mask, frame in primary.items():
    grouped = frame.groupby('seed')[all_cols]
    within = grouped.quantile(.75) - grouped.quantile(.25)
    normalized = within.div(scale.replace(0, np.nan), axis=1)
    regional = pd.DataFrame({r: normalized[region_cols[r]].median(axis=1) for r in REGIONS})
    summaries = {'Global measured panel': normalized[primary_cols].median(axis=1),
                 **{r: regional[r] for r in REGIONS},
                 'Equal-weight regions': regional.mean(axis=1)}
    for region, values in summaries.items():
        breadth_rows.extend({'mask': mask, 'seed': int(seed), 'region': region,
                             'normalized_breadth': value} for seed, value in values.items())
breadth = pd.DataFrame(breadth_rows)
breadth.to_csv(TAB / 'seed_level_breadth.csv', index=False)

def bootstrap_summary(part, index):
    wide = part.pivot(index='seed', columns='mask', values='normalized_breadth').dropna()
    a, b = wide.maskV1.to_numpy(), wide.maskV2.to_numpy()
    rng = np.random.default_rng(RNG_SEED + index)
    draws = rng.integers(0, len(wide), size=(N_BOOT, len(wide)))
    am = np.median(a[draws], axis=1); bm = np.median(b[draws], axis=1)
    draws_b = rng.integers(0, len(wide), size=(N_BOOT, len(wide)))
    independent = np.median(b[draws_b], axis=1) / am
    ratios = bm / am
    return {'region': part.region.iloc[0], 'seeds': len(wide),
            'maskV1_median_breadth': np.median(a), 'maskV2_median_breadth': np.median(b),
            'V2_over_V1': np.median(b) / np.median(a),
            'paired_CI_low': np.quantile(ratios, .025), 'paired_CI_high': np.quantile(ratios, .975),
            'independent_CI_low': np.quantile(independent, .025), 'independent_CI_high': np.quantile(independent, .975),
            'median_difference': np.median(b) - np.median(a),
            'difference_CI_low': np.quantile(bm - am, .025), 'difference_CI_high': np.quantile(bm - am, .975)}
order = ['Global measured panel', *REGIONS, 'Equal-weight regions']
breadth_summary = pd.DataFrame([bootstrap_summary(breadth[breadth.region.eq(r)], i) for i, r in enumerate(order)])
breadth_summary.to_csv(TAB / 'breadth_seed_bootstrap.csv', index=False)
print(breadth_summary.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.3), gridspec_kw={'width_ratios': [1.5, 1]}, layout='constrained')
labels = ['Global\nmeasured panel', 'Voltage\nsensors', 'Selectivity\nfilter', 'S6 / gate', 'Equal-weight\nregions']
sns.violinplot(data=breadth, x='region', y='normalized_breadth', hue='mask',
    order=order, hue_order=list(COLORS), palette=COLORS, split=True, inner='quart',
    cut=0, density_norm='width', ax=axes[0])
axes[0].set_xticks(range(5), labels, fontsize=9)
axes[0].set(xlabel='', ylabel='Normalized breadth per seed', title='A  Breadth across AF2 models within each seed')
axes[0].legend(title='', frameon=False, loc='upper right')
s = breadth_summary
axes[1].errorbar(s.V2_over_V1, np.arange(len(s)),
    xerr=np.vstack([s.V2_over_V1-s.paired_CI_low, s.paired_CI_high-s.V2_over_V1]),
    fmt='o', color=COLORS['maskV2'], capsize=4)
axes[1].axvline(1, color='.5', ls='--', lw=1)
axes[1].set_yticks(range(5), [x.replace('\n', ' ') for x in labels], fontsize=9)
axes[1].invert_yaxis()
axes[1].set(xlabel='maskV2 / maskV1 breadth (95% seed-bootstrap CI)',
            title='B  Relative sampling breadth')
savefig(fig, '01_global_and_regional_breadth')

## Coordinate-level distribution summaries

Ensemble IQR, median absolute deviation (MAD), SD, median shift, and Wasserstein distance are calculated across the 500 matched trajectory representatives for every invariant coordinate. This ensemble-IQR ratio is distinct from the seed-level breadth ratio above. Ratios with zero maskV1 denominator are undefined (NaN), not inflated with a pseudocount. Region summaries are descriptive medians across correlated coordinates, not independent-coordinate significance tests.

In [ ]:
def distance_summary(pair):
    a, b = [pair[k][all_cols] for k in COLORS]
    q1, q2 = a.quantile([.25, .5, .75]), b.quantile([.25, .5, .75])
    iqr1, iqr2 = q1.loc[.75]-q1.loc[.25], q2.loc[.75]-q2.loc[.25]
    mad1, mad2 = (a-q1.loc[.5]).abs().median(), (b-q2.loc[.5]).abs().median()
    stats = pd.DataFrame({'maskV1_median_A': q1.loc[.5], 'maskV2_median_A': q2.loc[.5],
        'median_shift_A': q2.loc[.5]-q1.loc[.5], 'maskV1_IQR_A': iqr1, 'maskV2_IQR_A': iqr2,
        'IQR_ratio': iqr2/iqr1.replace(0, np.nan), 'maskV1_MAD_A': mad1, 'maskV2_MAD_A': mad2,
        'MAD_ratio': mad2/mad1.replace(0, np.nan), 'maskV1_SD_A': a.std(), 'maskV2_SD_A': b.std(),
        'SD_ratio': b.std()/a.std().replace(0, np.nan),
        'Wasserstein_A': [wasserstein_distance(a[c], b[c]) for c in all_cols]})
    return coordinates.merge(stats.rename_axis('coordinate').reset_index(), on='coordinate', validate='one_to_one')
stats = distance_summary(primary)
stats.to_csv(TAB / 'coordinate_distribution_statistics.csv', index=False)
regional_stats = stats.groupby(['measurement', 'region']).agg(
    coordinates=('coordinate', 'size'), median_IQR_ratio=('IQR_ratio', 'median'),
    median_SD_ratio=('SD_ratio', 'median'), median_MAD_ratio=('MAD_ratio', 'median'),
    median_signed_shift_A=('median_shift_A', 'median'), median_Wasserstein_A=('Wasserstein_A', 'median')).reset_index()
regional_stats.to_csv(TAB / 'regional_distribution_statistics.csv', index=False)
print(regional_stats.round(3).to_string(index=False))

## Figure 3-style regional distance distributions

The landmarks are G375 at the selectivity filter, A402 at the gate, and F236–R289 / F236–R308 in the VSD. Ring ranks run from the smallest to largest of the six intersubunit spans; VSD ranks run across the four homologous subunits. Violin halves use the same coordinate and units for both masks. Quartile marks show the median and IQR. These panels use the matched recycle-10 cohort from both final filtered ensembles. Black diamonds mark WT 8SD3 and purple pentagons mark L403A 8SDA experimental structures, calculated from the local PDB with the same rank transformation and CSV-to-rat numbering map; they are geometric reference values, not QC thresholds.

If any constituent distance is unavailable in the experimental structure, its entire ranked reference set is left missing; ranks are never reassigned from a partial set.
The complete six-structure experimental overlay is exported in Figure 07 and the all-coordinate reference table.


In [ ]:
from shared.kv21_experimental import coordinate_model, ca_coordinate
experimental_refs = {}
for reference_id in ['8SD3', '8SDA', '9O10', '9O11', '9O12', '9O13']:
    reference_path = ROOT / f'kv21/experimental/{reference_id}.pdb'
    assert reference_path.is_file()
    reference_model = coordinate_model(reference_id, repo_root=ROOT)
    reference_rows = []
    for row in coordinates[coordinates.measurement.eq('CA')].itertuples():
        distances = []
        for column in row.source_columns.split(';'):
            ch1, res1, n1, ch2, res2, n2 = ca_pattern.fullmatch(column).groups()
            try:
                left = ca_coordinate(reference_model, ch1, int(n1)-2)
                right = ca_coordinate(reference_model, ch2, int(n2)-2)
                distances.append(float(np.linalg.norm(left-right)))
            except KeyError:
                distances.append(np.nan)
        reference_rows.append({'coordinate': row.coordinate, 'structure': reference_id,
                               'distance_A': sorted(distances)[row.rank-1] if np.isfinite(distances).all() else np.nan,
                               'available_source_distances': int(np.isfinite(distances).sum()),
                               'required_source_distances': len(distances),
                               'pdb_sha256': sha256(reference_path)})
    experimental_refs[reference_id] = pd.DataFrame(reference_rows).set_index('coordinate')
experimental = experimental_refs['8SD3']
experimental_long = pd.concat(experimental_refs.values()).reset_index()
experimental_long.to_csv(TAB / 'experimental_reference_distances_8SD3_8SDA.csv', index=False)


In [ ]:
def split_panel(ax, rows, title):
    records = []
    labels = []
    for row in rows.itertuples():
        label = f'{row.pair}\nrank {row.rank}'
        labels.append(label)
        for mask, frame in primary.items():
            records.extend({'coordinate': label, 'Distance (Å)': v, 'mask': mask}
                           for v in frame[row.coordinate])
    sns.violinplot(data=pd.DataFrame(records), x='coordinate', y='Distance (Å)', hue='mask',
        order=labels, hue_order=list(COLORS), palette=COLORS, split=True, inner='quart',
        cut=0, density_norm='width', ax=ax, linewidth=.7)
    ax.set(xlabel='', title=title)
    ax.tick_params(axis='x', labelsize=8)
    reference_styles = {'8SD3': ('#202020', 'D', -0.15), '8SDA': ('#7A3E9D', 'P', -0.09), '9O10': ('#17807E', '^', -0.03), '9O11': ('#2C63A5', 'v', 0.03), '9O12': ('#D97925', 'X', 0.09), '9O13': ('#B33C4A', 'H', 0.15)}
    for x, row in enumerate(rows.itertuples()):
        for reference_id, (color, marker, offset) in reference_styles.items():
            ref = experimental_refs[reference_id]
            if row.coordinate in ref.index and pd.notna(ref.loc[row.coordinate, 'distance_A']):
                ax.scatter(x + offset, ref.loc[row.coordinate, 'distance_A'], marker=marker,
                           color=color, s=28, zorder=8)
    ax.legend_.remove()

fig, axes = plt.subplots(3, 1, figsize=(13, 12), layout='constrained')
ca = coordinates[coordinates.measurement.eq('CA')]
landmarks = [ca[ca.pair.eq('G375')], ca[ca.pair.eq('A402')],
             ca[ca.pair.isin(['R289–F236', 'R308–F236'])]]
for ax, rows, title in zip(axes, landmarks,
    ['A  Selectivity filter: G375 ring', 'B  Intracellular gate: A402 ring',
     'C  Voltage sensors: F236–R289 and F236–R308']):
    assert len(rows)
    split_panel(ax, rows.sort_values(['pair', 'rank']), title)
axes[0].legend(handles=[Patch(color=c, label=m) for m, c in COLORS.items()] + [
    matplotlib.lines.Line2D([], [], color='#202020', marker='D', ls='', label='8SD3 WT'),
    matplotlib.lines.Line2D([], [], color='#7A3E9D', marker='P', ls='', label='8SDA L403A'),
    matplotlib.lines.Line2D([], [], color='#17807E', marker='^', ls='', label='9O10'),
    matplotlib.lines.Line2D([], [], color='#2C63A5', marker='v', ls='', label='9O11'),
    matplotlib.lines.Line2D([], [], color='#D97925', marker='X', ls='', label='9O12'),
    matplotlib.lines.Line2D([], [], color='#B33C4A', marker='H', ls='', label='9O13')],
    frameon=False, loc='upper left', ncol=4, fontsize=8)
savefig(fig, '02_figure3_landmark_distance_distributions')

## Full filtered survivor comparison

This companion Figure 3-style plot uses every retained filtered recycle-10 structure in each mask. It does not require a shared `(seed, AF2 model)` key, so ensemble sizes remain unbalanced and represent the full survivor pools.


In [ ]:
all_survivors = {mask: frame[frame.recycle.eq(10)].copy() for mask, frame in transformed.items()}
survivor_counts = pd.DataFrame([{'mask': mask, 'recycle': 10, 'structures': len(frame),
    'seeds': frame.seed.nunique(), 'trajectories': frame[['seed','model']].drop_duplicates().shape[0]}
    for mask, frame in all_survivors.items()])
survivor_counts.to_csv(TAB / 'full_survivor_r10_counts.csv', index=False)
print(survivor_counts.to_string(index=False))

def split_panel_all(ax, rows, title):
    records = []
    labels = []
    for row in rows.itertuples():
        label = f'{row.pair}\nrank {row.rank}'
        labels.append(label)
        for mask, frame in all_survivors.items():
            records.extend({'coordinate': label, 'Distance (Å)': v, 'mask': mask}
                           for v in frame[row.coordinate])
    sns.violinplot(data=pd.DataFrame(records), x='coordinate', y='Distance (Å)', hue='mask',
        order=labels, hue_order=list(COLORS), palette=COLORS, split=True, inner='quart',
        cut=0, density_norm='width', ax=ax, linewidth=.7)
    ax.set(xlabel='', title=title)
    ax.tick_params(axis='x', labelsize=8)
    reference_styles = {'8SD3': ('#202020', 'D', -0.15), '8SDA': ('#7A3E9D', 'P', -0.09),
                        '9O10': ('#17807E', '^', -0.03), '9O11': ('#2C63A5', 'v', 0.03),
                        '9O12': ('#D97925', 'X', 0.09), '9O13': ('#B33C4A', 'H', 0.15)}
    for x, row in enumerate(rows.itertuples()):
        for reference_id, (color, marker, offset) in reference_styles.items():
            ref = experimental_refs[reference_id]
            if row.coordinate in ref.index and pd.notna(ref.loc[row.coordinate, 'distance_A']):
                ax.scatter(x + offset, ref.loc[row.coordinate, 'distance_A'], marker=marker,
                           color=color, s=28, zorder=8)
    ax.legend_.remove()

fig, axes = plt.subplots(3, 1, figsize=(13, 12), layout='constrained')
for ax, rows, title in zip(axes, landmarks,
    ['A  Selectivity filter: G375 ring — all survivors',
     'B  Intracellular gate: A402 ring — all survivors',
     'C  Voltage sensors: F236–R289 and F236–R308 — all survivors']):
    assert len(rows)
    split_panel_all(ax, rows.sort_values(['pair', 'rank']), title)
axes[0].legend(handles=[Patch(color=c, label=m) for m, c in COLORS.items()] + [
    matplotlib.lines.Line2D([], [], color='#202020', marker='D', ls='', label='8SD3 WT'),
    matplotlib.lines.Line2D([], [], color='#7A3E9D', marker='P', ls='', label='8SDA L403A'),
    matplotlib.lines.Line2D([], [], color='#17807E', marker='^', ls='', label='9O10'),
    matplotlib.lines.Line2D([], [], color='#2C63A5', marker='v', ls='', label='9O11'),
    matplotlib.lines.Line2D([], [], color='#D97925', marker='X', ls='', label='9O12'),
    matplotlib.lines.Line2D([], [], color='#B33C4A', marker='H', ls='', label='9O13')],
    frameon=False, loc='upper left', ncol=4, fontsize=8)
savefig(fig, '08_figure3_all_filtered_survivors')


## S6 profile and selectivity-filter distributions

For each complete ring, the largest of the six Cα spans is plotted. The empirical cumulative distribution functions (ECDFs) show the observed values without a smoothing bandwidth. S6 labels are V398, I401, A402, and L403. Distal I405/V409 are unavailable in the new files.

In [ ]:
rings = ca[ca.region.ne('Voltage sensors') & ca['rank'].eq(6)].sort_values(['region', 'csv_residue_1'])
fig, axes = plt.subplots(2, 4, figsize=(15, 7.8), layout='constrained')
for row_index, region in enumerate([REGIONS[2], REGIONS[1]]):
    for ax, row in zip(axes[row_index], rings[rings.region.eq(region)].itertuples()):
        for mask, frame in primary.items():
            x = np.sort(frame[row.coordinate].to_numpy())
            ax.step(x, np.arange(1, len(x)+1)/len(x), where='post', color=COLORS[mask], label=mask, lw=1.8)
        ax.axvline(experimental_refs['8SD3'].loc[row.coordinate, 'distance_A'], color='#202020', ls=':', lw=1.4, label='8SD3 WT')
        ax.axvline(experimental_refs['8SDA'].loc[row.coordinate, 'distance_A'], color='#7A3E9D', ls='--', lw=1.2, label='8SDA L403A')
        for reference_id, color in [('9O10','#17807E'),('9O11','#2C63A5'),('9O12','#D97925'),('9O13','#B33C4A')]:
            ax.axvline(experimental_refs[reference_id].loc[row.coordinate, 'distance_A'], color=color, ls='-.', lw=1.0, alpha=.9, label=reference_id)
        result = stats[stats.coordinate.eq(row.coordinate)].iloc[0]
        ax.set(title=f'{row.pair}: IQR ratio {result.IQR_ratio:.2f}', xlabel='Maximum Cα ring span (Å)',
               ylabel='Cumulative fraction' if ax in axes[:, 0] else '', ylim=(0, 1.03))
        ax.text(.03, .92, 'S6 / gate' if row_index == 0 else 'Selectivity filter', transform=ax.transAxes, fontsize=9)
axes[0, 0].legend(frameon=False, loc='lower right', fontsize=9)
savefig(fig, '03_s6_and_filter_ring_distributions')

def vsd_heatmap(values, field):
    return values.pivot(index='paper_residue_1', columns='paper_residue_2', values=field).reindex(
        index=['R289','R290','R296','R299','R302','K305','R308'],
        columns=['E176','D221','E229','C232','F236','E239','D262','E277'])
vsd = stats[stats.measurement.eq('CA') & stats.region.eq(REGIONS[0])].groupby(
    ['paper_residue_1', 'paper_residue_2'], as_index=False).agg(
        IQR_ratio=('IQR_ratio', 'median'), median_shift_A=('median_shift_A', 'median'))
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), layout='constrained')
sns.heatmap(np.log2(vsd_heatmap(vsd, 'IQR_ratio')), cmap='vlag', center=0,
            annot=vsd_heatmap(vsd, 'IQR_ratio'), fmt='.2f', ax=axes[0],
            cbar_kws={'label': 'log₂(maskV2 / maskV1 IQR)'}, annot_kws={'size': 8})
sns.heatmap(vsd_heatmap(vsd, 'median_shift_A'), cmap='vlag', center=0, annot=True, fmt='.1f',
            ax=axes[1], cbar_kws={'label': 'maskV2 − maskV1 median distance (Å)'}, annot_kws={'size': 8})
for ax in axes:
    ax.set(xlabel='VSD partner (paper numbering)', ylabel='S4 position (paper numbering)')
axes[0].set_title('A  VSD breadth (numbers: IQR ratios; color: log₂)')
axes[1].set_title('B  VSD position: median shift over four subunit ranks')
savefig(fig, '04_voltage_sensor_distance_maps')

## Dependence on AF2 model parameterization

Strong differences between model parameterizations can broaden the combined ensemble. The panels below separate the the available retained AF2 models, with 100 matched seeds per model. The dotted line is the corresponding WT 8SD3 maximum Cα ring span. Large departures from this reference are not automatically evidence of biologically accessible states; distance tables alone cannot establish whether subunits remain correctly assembled.

The accompanying tables summarize coordinate IQR ratios within each parameterization and recompute the global/regional seed-breadth ratios after omitting each AF2 model. Omission analyses use the original pooled scales and IQRs over the remaining four models, without resampling or a significance test.

In [ ]:
model_rows = []
for model in sorted(primary['maskV1'].model.unique()):
    pair = {mask: frame[frame.model.eq(model)] for mask, frame in primary.items()}
    table = distance_summary(pair)
    for region in REGIONS:
        part = table[table.measurement.eq('CA') & table.region.eq(region)]
        model_rows.append({'AF2_model': model, 'region': region, 'seeds': pair['maskV1'].seed.nunique(),
                           'median_coordinate_IQR_ratio': part.IQR_ratio.median(),
                           'median_coordinate_shift_A': part.median_shift_A.median()})
model_summary = pd.DataFrame(model_rows)
model_summary.to_csv(TAB / 'AF2_model_regional_statistics.csv', index=False)
print(model_summary.round(3).to_string(index=False))

model_landmarks = {'G375 selectivity filter': 'CA_GLY377_GLY377_rank6',
                   'A402 intracellular gate': 'CA_ALA404_ALA404_rank6'}
model_quantiles = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for ax, (title, column) in zip(axes, model_landmarks.items()):
    panel = pd.concat([frame[['model', column]].rename(columns={column:'distance_A'}).assign(mask=mask)
                       for mask, frame in primary.items()], ignore_index=True)
    sns.violinplot(data=panel, x='model', y='distance_A', hue='mask', split=True,
        hue_order=list(COLORS), palette=COLORS, inner='quart', cut=0, density_norm='width', ax=ax)
    ax.axhline(experimental_refs['8SD3'].loc[column, 'distance_A'], color='#202020', ls=':', lw=1.4)
    ax.axhline(experimental_refs['8SDA'].loc[column, 'distance_A'], color='#7A3E9D', ls='--', lw=1.2)
    for reference_id, color in [('9O10','#17807E'),('9O11','#2C63A5'),('9O12','#D97925'),('9O13','#B33C4A')]:
        ax.axhline(experimental_refs[reference_id].loc[column, 'distance_A'], color=color, ls='-.', lw=1.0)
    ax.set(title=title, xlabel='AF2 model parameterization', ylabel='Maximum Cα ring span (Å)')
    ax.legend_.remove()
    for (mask, model), group in panel.groupby(['mask', 'model']):
        q = group.distance_A.quantile([.25, .5, .75])
        model_quantiles.append({'landmark': title, 'mask': mask, 'AF2_model': model, 'seeds': len(group),
                                'q25_A': q.loc[.25], 'median_A': q.loc[.5], 'q75_A': q.loc[.75]})
fig.legend(handles=[Patch(color=c, label=m) for m, c in COLORS.items()] + [
    matplotlib.lines.Line2D([], [], color='#202020', ls=':', label='8SD3 WT'),
    matplotlib.lines.Line2D([], [], color='#7A3E9D', ls='--', label='8SDA L403A'),
    matplotlib.lines.Line2D([], [], color='#17807E', ls='-.', label='9O10'),
    matplotlib.lines.Line2D([], [], color='#2C63A5', ls='-.', label='9O11'),
    matplotlib.lines.Line2D([], [], color='#D97925', ls='-.', label='9O12'),
    matplotlib.lines.Line2D([], [], color='#B33C4A', ls='-.', label='9O13')],
    loc='outside upper center', ncol=3, frameon=False)
savefig(fig, '06_AF2_model_resolved_ring_distributions')
model_quantiles = pd.DataFrame(model_quantiles)
model_quantiles.to_csv(TAB / 'AF2_model_landmark_quantiles.csv', index=False)

omission_rows = []
for omitted in sorted(primary['maskV1'].model.unique()):
    summaries = {}
    for mask, frame in primary.items():
        group = frame[frame.model.ne(omitted)].groupby('seed')[primary_cols]
        normalized = (group.quantile(.75)-group.quantile(.25)).div(scale[primary_cols], axis=1)
        summaries[mask] = {'Global measured panel': normalized.median(axis=1).median(),
                          **{r: normalized[region_cols[r]].median(axis=1).median() for r in REGIONS}}
    for region in ['Global measured panel', *REGIONS]:
        omission_rows.append({'omitted_AF2_model': omitted, 'region': region,
            'seed_breadth_ratio_V2_over_V1': summaries['maskV2'][region]/summaries['maskV1'][region]})
omissions = pd.DataFrame(omission_rows)
omissions.to_csv(TAB / 'leave_one_AF2_model_out_breadth.csv', index=False)
print(omissions.pivot(index='region', columns='omitted_AF2_model', values='seed_breadth_ratio_V2_over_V1').round(2).to_string())


## Recycle and published-QC sensitivity

Recycle-resolved IQR ratios test whether the pattern depends on choosing recycle 10. Each recycle is summarized separately over matched trajectories; successive recycles are not independent evidence. An additional analysis replaces each trajectory by its median over recycles 1–10.

The published maskV1 QC subset is compared with maskV2 at exactly the same seed, model, and recycle as the latest retained maskV1 snapshot. This controls trajectory composition and recycle depth but **does not apply structural QC to maskV2**. It is a reference-cohort sensitivity, not an equally filtered comparison.

In [ ]:
sensitivity_rows = []
available_recycles = sorted(set(transformed['maskV1'].recycle) & set(transformed['maskV2'].recycle))
for recycle in available_recycles:
    pair = matched_pair(transformed, recycle)
    current = distance_summary(pair)
    for region in REGIONS:
        part = current[current.measurement.eq('CA') & current.region.eq(region)]
        sensitivity_rows.append({'analysis': f'recycle {recycle}', 'recycle': recycle,
            'region': region, 'trajectories': len(pair['maskV1']),
            'median_IQR_ratio': part.IQR_ratio.median(), 'median_shift_A': part.median_shift_A.median()})

trajectory_medians = {}
for mask, frame in transformed.items():
    trajectory_medians[mask] = frame[frame.recycle.between(1,10)].groupby(['seed', 'model'], as_index=False)[all_cols].median()
median_stats = distance_summary(trajectory_medians)
median_stats.to_csv(TAB / 'trajectory_median_distribution_statistics.csv', index=False)

qc_latest = qc_transformed.sort_values('recycle').groupby(['seed', 'model'], as_index=False).tail(1)
keys = ['seed', 'model', 'recycle']
v2_indexed = transformed['maskV2'].set_index(keys)
# Keep only QC reference snapshots that exist in both filtered ensembles.
qc_latest = qc_latest.merge(transformed['maskV2'][keys].drop_duplicates(), on=keys, how='inner')
qc_keys = pd.MultiIndex.from_frame(qc_latest[keys])
qc_pair = {'maskV1': qc_latest.reset_index(drop=True),
           'maskV2': v2_indexed.loc[qc_keys].reset_index()}
qc_stats = distance_summary(qc_pair)
qc_stats.to_csv(TAB / 'published_maskV1_QC_reference_statistics.csv', index=False)
for label, table, count in [
    ('trajectory median r1–r10', median_stats, len(trajectory_medians['maskV1'])),
    ('filtered V1 / filtered V2, matched snapshots', qc_stats, len(qc_latest))]:
    for region in REGIONS:
        part = table[table.measurement.eq('CA') & table.region.eq(region)]
        sensitivity_rows.append({'analysis': label, 'recycle': np.nan, 'region': region,
            'trajectories': count, 'median_IQR_ratio': part.IQR_ratio.median(),
            'median_shift_A': part.median_shift_A.median()})
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(TAB / 'cohort_and_recycle_sensitivity.csv', index=False)
print(sensitivity[sensitivity.recycle.isna()].round(3).to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), layout='constrained')
for ax, region in zip(axes, REGIONS):
    part = sensitivity[sensitivity.region.eq(region) & sensitivity.recycle.notna()]
    ax.plot(part.recycle, part.median_IQR_ratio, marker='o', color=COLORS['maskV2'])
    ax.axhline(1, color='.5', lw=1, ls='--')
    ax.set(title=region, xlabel='Recycle', ylabel='Median coordinate IQR ratio', xticks=range(0,11,2))
savefig(fig, '05_recycle_dependence')

## Results and reproducibility

The statements below are calculated from the current input files. A ratio above 1 means broader sampling under maskV2 for the stated estimator; below 1 means narrower sampling. Increased spread does not by itself indicate improved structural accuracy. Comparisons to the published QC subset remain confounded by the absence of equivalent maskV2 QC. The available panel cannot assess distal S6, the full pore–VSD interface, or ensemble RMSF.

In [ ]:
# Full experimental reference coverage: every available coordinate across all six structures.
exp_all = experimental_long.merge(coordinates[['coordinate','measurement','region','pair','rank']], on='coordinate', how='left')
exp_all.to_csv(TAB / 'experimental_reference_distances_all_coordinates.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), layout='constrained')
for ax, (measurement, region) in zip(axes, [('CA', REGIONS[0]), ('CA', REGIONS[1]), ('CA', REGIONS[2])]):
    subset = exp_all[(exp_all.measurement.eq(measurement)) & (exp_all.region.eq(region))]
    wide = subset.pivot(index='structure', columns='coordinate', values='distance_A')
    sns.heatmap(wide, ax=ax, cmap='viridis', cbar=True, xticklabels=False,
                yticklabels=True, linewidths=0.0, cbar_kws={'label':'Distance (Å)'})
    ax.set_title(f'{measurement}: {region} — all experimental coordinates')
    ax.set_xlabel('Coordinate index')
    ax.set_ylabel('Experimental structure')
savefig(fig, '07_all_experimental_reference_distances')


In [ ]:
lines = []
for row in breadth_summary.itertuples():
    lines.append(f'- **{row.region}:** maskV2/maskV1 seed-breadth ratio '
                 f'**{row.V2_over_V1:.2f}** (paired seed-bootstrap 95% CI '
                 f'{row.paired_CI_low:.2f}–{row.paired_CI_high:.2f}).')
print('\n'.join(line.replace('**','') for line in lines))
filter_models = model_quantiles[model_quantiles.landmark.eq('G375 selectivity filter') & model_quantiles['mask'].eq('maskV2')]
model_text = '; '.join(f"model {int(row.AF2_model)}: {row.median_A:.1f} Å" for row in filter_models.itertuples())
ref = experimental.loc['CA_GLY377_GLY377_rank6', 'distance_A']
print(f'Model dependence: maskV2 median G375 maximum spans are {model_text}; WT 8SD3 reference is {ref:.1f} Å.')
ring_stats = stats[(stats.measurement.eq('CA')) & stats.region.eq(REGIONS[2]) & stats['rank'].eq(6)]
print(ring_stats[['pair','maskV1_median_A','maskV2_median_A','maskV1_IQR_A','maskV2_IQR_A',
                    'IQR_ratio','SD_ratio','median_shift_A']].round(3).to_string(index=False))

# Verify key invariants and figure source data before reporting completion.
# Reordering source subunit measurements cannot change the ranked coordinates.
probe = raw['maskV2'].iloc[:8].copy()
for sources in coordinates.source_columns.unique():
    columns = sources.split(';')
    probe[columns] = probe[columns[::-1]].to_numpy()
np.testing.assert_array_equal(transform(probe)[all_cols], transform(raw['maskV2'].iloc[:8])[all_cols])
assert len(raw['maskV2']) > 0
assert sum(len(v) for v in region_cols.values()) == len(primary_cols)
assert set(primary_cols).issubset(valid_cols)
assert np.isfinite(breadth.normalized_breadth).all()
assert len(breadth_summary) == 5
assert not stats.coordinate.duplicated().any()
assert (stats.Wasserstein_A >= 0).all()
assert primary['maskV1'][['seed','model']].equals(primary['maskV2'][['seed','model']])
parameters = {
    'bootstrap_replicates': N_BOOT, 'random_seed': RNG_SEED,
    'primary_recycle': 10, 'primary_filtering': 'all_ok3 + G377 structural QC + 27 A pore-VSD interface QC + stable-core alignment QC for both masks',
    'raw_maskV1_cached_snapshot': 'filtered v1 source resolved by data_access',
    'normalization': 'pooled coordinate IQR across equal matched r10 ensembles',
    'seed_breadth': 'median coordinate-normalized IQR over the available retained AF2 models per seed',
    'seed_bootstrap': 'paired nominal seed blocks; independent-seed sensitivity also reported',
    'chain_treatment': 'sorted four intrachain VSD or six complete same-residue ring distances per snapshot',
    'missing_S6_CSV_levels': [407, 411],
    'experimental_reference': {'PDB': '8SD3', 'sha256': sha256(reference_path), 'mapping': 'CSV residue minus 2'},
    'versions': {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
                 'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__, 'seaborn': sns.__version__},
}
(OUT / 'analysis_parameters.json').write_text(json.dumps(parameters, indent=2) + '\n')
print('All consistency checks passed.')
print('Combined CSV:', (OUT / 'Kv21_WT_maskV2_combined_distances.csv.gz').relative_to(ROOT))
print('Tables:', TAB.relative_to(ROOT))
print('Figures:', FIG.relative_to(ROOT))